# stack-vs-cat composite — cx9: repeat-broadcast triangle vertices then cat along the d-axis

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat-broadcast`, `stack-vs-cat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "stack-vs-cat"
DD_ATOM_IDS = ["einops-repeat-broadcast", "stack-vs-cat"]
DD_SUBTOPICS = ["Einops: Repeat-as-broadcast", "PyTorch: stack vs cat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Some ray-triangle solvers want the three triangle vertices in a single concatenated 9-vector per (ray, triangle) slot — `(NR, NT, 9)` where the last axis is `[Ax,Ay,Az,Bx,By,Bz,Cx,Cy,Cz]`.

Build it two ways atoms compose:
  1. `einops-repeat-broadcast` inserts the ray-axis into each vertex `(NT, 3) -> (NR, NT, 3)`.
  2. `stack-vs-cat` picks `torch.cat(..., dim=-1)` — we're EXTENDING an existing 3-axis to 9, NOT inserting a new one. Picking stack here would give the wrong shape `(NR, NT, 3, 3)`.

The rule from cx8 reverses here: same total axes in the target → cat, not stack.

### Composite Exercise — repeat-broadcast triangle vertices then cat along the d-axis

**Atoms exercised together**: `einops-repeat-broadcast`, `stack-vs-cat`

Implement `cx9_broadcast_and_concat_vertices(triangles, NR)` that takes triangle vertices `triangles: (NT, 3, 3)` (axis 1 = vertex A/B/C, axis 2 = xyz) and the number of rays `NR`, and returns a `(NR, NT, 9)` tensor where each slot is the concatenated 9-vector `[A, B, C]`.

1. **Split** triangles into A, B, C — three `(NT, 3)` tensors.
2. **Repeat-broadcast** each into `(NR, NT, 3)` with `repeat(..., 't d -> r t d', r=NR)`. These are stride-0 views.
3. **Stack vs cat**: target last axis size 9 = 3 + 3 + 3 — that's EXTENDING the existing d-axis. Use `torch.cat([..., ..., ...], dim=-1)` (NOT stack).

Return shape `(NR, NT, 9)`. The test asserts the slicing `[..., :3] == A`, `[..., 3:6] == B`, `[..., 6:] == C` for every (r, t).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx9_broadcast_and_concat_vertices(triangles, NR):
    raise NotImplementedError

def _test_cx9():
    # atom-coverage: enforce that the solution actually uses einops.repeat
    # (not .expand() or t.broadcast_to or a full-copy tensor). Without this
    # check the value-tests pass with .expand and the einops-repeat atom
    # claim is fig-leaf.
    import inspect
    _src = inspect.getsource(cx9_broadcast_and_concat_vertices)
    assert 'repeat(' in _src, 'solution must use einops.repeat (not .expand/.broadcast_to/full-copy)'
    NT = 4
    triangles = t.randn(NT, 3, 3)  # (NT, vertex_abc, xyz)
    NR = 5
    out = cx9_broadcast_and_concat_vertices(triangles, NR)
    assert tuple(out.shape) == (NR, NT, 9), f'shape: {tuple(out.shape)}'
    # Slot layout check.
    for r in range(NR):
        for ti in range(NT):
            assert t.equal(out[r, ti, :3], triangles[ti, 0]), 'first 3 must be vertex A'
            assert t.equal(out[r, ti, 3:6], triangles[ti, 1]), 'middle 3 must be vertex B'
            assert t.equal(out[r, ti, 6:], triangles[ti, 2]), 'last 3 must be vertex C'

    # Case B: hand-check on simple tensor.
    tri2 = t.tensor([[[1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0],
                      [7.0, 8.0, 9.0]]])  # (NT=1, V=3, D=3)
    out2 = cx9_broadcast_and_concat_vertices(tri2, NR=2)
    expected_row = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
    assert tuple(out2.shape) == (2, 1, 9)
    assert t.equal(out2[0, 0], expected_row)
    assert t.equal(out2[1, 0], expected_row)

    # Case C: NEGATIVE — stack would give (NR, NT, 3, 3), not (NR, NT, 9).
    assert out.ndim == 3, f'expected 3-D (NR,NT,9), got {out.ndim}-D — did you use stack?'

    # Case D: realistic scale.
    tri3 = t.randn(50, 3, 3)
    out3 = cx9_broadcast_and_concat_vertices(tri3, NR=100)
    assert tuple(out3.shape) == (100, 50, 9)
    _dd_passed.add('cx9')

_test_cx9()

<details><summary>Show solution — cx9</summary>

```python
def cx9_broadcast_and_concat_vertices(triangles, NR):
    A = triangles[:, 0]  # (NT, 3)
    B = triangles[:, 1]
    C = triangles[:, 2]
    # Atom A (einops-repeat-broadcast): insert ray-axis as stride-0 view on each vertex.
    A_b = repeat(A, 't d -> r t d', r=NR)
    B_b = repeat(B, 't d -> r t d', r=NR)
    C_b = repeat(C, 't d -> r t d', r=NR)
    # Atom B (stack-vs-cat): we want the LAST axis extended from 3 to 9 — that's cat, not stack.
    return t.cat([A_b, B_b, C_b], dim=-1)
```

stack-vs-cat decision tree: are you ADDING an axis or GROWING one?
  - Target has more axes than each input  → stack (inserts axis).
  - Target has same axes as each input    → cat (extends an axis).

Here each `A_b`/`B_b`/`C_b` is (NR, NT, 3) and the target is (NR, NT, 9) — same axes, 9 = 3+3+3 → cat along the last axis. stack would have given (NR, NT, 3, 3).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx9',
        'subtopics': ["Einops: Repeat-as-broadcast", "PyTorch: stack vs cat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()